# Homologación de datos electorales — Cárdenas, Tabasco
PELO 2023-2024: Gubernatura, Diputaciones y Ayuntamientos

Objetivo: consolidar los 7 archivos .xlsx (sección electoral × casilla)
en una sola tabla normalizada lista para el geovisor.

In [15]:
import pandas as pd
from pathlib import Path


pd.set_option('display.max_columns', None)

AttributeError: module 'pandas' has no attribute 'set_option'

In [12]:
TABLAS    = Path("../datos/insumos/tablas")
PROCESADOS = Path("../datos/procesados")
PROCESADOS.mkdir(exist_ok=True)

## 1. Exploración rápida de cada archivo

In [13]:
archivos = sorted(TABLAS.glob("*.xlsx"))

for f in archivos:
    df = pd.read_excel(f)
    print(f"{'='*60}")
    print(f"Archivo : {f.name}")
    print(f"Filas   : {len(df)}  |  Columnas: {len(df.columns)}")
    print(f"Cols    : {list(df.columns)}")
    print()

AttributeError: module 'pandas' has no attribute 'read_excel'

## 2. Esquema objetivo

 Columnas comunes a todos los archivos (nombres normalizados):

 seccion        — sección electoral (str, 4 dígitos con cero a la izquierda)
 casilla        — nombre de la casilla (BÁSICA, CONTIGUA 01, VOTO ANTICIPADO…)
 tipo_eleccion  — 'gubernatura' | 'diputaciones' | 'ayuntamiento'
 distrito       — 'DT-01'…'DT-03' | 'DTO-01'…'DTO-03' | None (ayuntamiento cubre los 3)

 Votos por partido (pueden ser NaN si el partido no participó en esa elección):
 PAN, PRI, PRD, PVEM, PT, MC, MORENA, CAND_IND

 Totales de coalición (cuando aplica):
 total_pvem_pt_morena   — TOTAL PVEM, PT Y MORENA  (solo gubernatura)
 total_pan_pri          — TOTAL PAN Y PRI           (solo gubernatura)
 total_pvem_morena      — TOTAL PVEM Y MORENA       (gubernatura + dto01)

 Resumen de votación:
 votos_validos, cand_no_reg, votos_nulos, total_votos, lista_nominal, participacion

 Banderas (cuando aplica):
 anulada, modificada

## 3. Funciones de normalización

In [ ]:
# Mapeo de columnas crudas → nombre normalizado.
# Clave: nombre crudo (tal como aparece en el xlsx, ignorando mayúsculas/minúsculas).
# Valor: nombre objetivo en el esquema unificado.

RENAME_MAP = {
    # sección
    "sección electoral"                               : "seccion",
    "seccion electoral"                               : "seccion",  # sin tilde
    # casilla
    "casilla"                                         : "casilla",
    # partidos
    "pan"                                             : "PAN",
    "pri"                                             : "PRI",
    "prd"                                             : "PRD",
    "pvem"                                            : "PVEM",
    "pt"                                              : "PT",
    "mc"                                              : "MC",
    "morena"                                          : "MORENA",
    "c. ind. juan carlos guzman correa"               : "CAND_IND",
    # coaliciones — solo los TOTALES (las sub-columnas se descartan)
    "total pvem, pt y morena"                         : "total_pvem_pt_morena",
    "total pvem y morena"                             : "total_pvem_morena",
    "total pan y pri"                                 : "total_pan_pri",
    # resumen
    "número de votos válidos"                         : "votos_validos",
    "numero de votos validos"                         : "votos_validos",
    "número de votos candidaturas no registradas"     : "cand_no_reg",
    "numero de votos candidaturas no registradas"     : "cand_no_reg",
    "número de votos a candidaturas no registradas"   : "cand_no_reg",
    "numero de votos a candidaturas no registradas"   : "cand_no_reg",
    "número de votos nulos"                           : "votos_nulos",
    "numero de votos nulos"                           : "votos_nulos",
    "total de votos"                                  : "total_votos",
    "lista nominal"                                   : "lista_nominal",
    "% participación ciudadana"                       : "participacion",
    "% participacion ciudadana"                       : "participacion",
    "participación ciudadana"                         : "participacion",
    "participacion ciudadana"                         : "participacion",
    # banderas
    "anulada"                                         : "anulada",
    "modificada"                                      : "modificada",
}

# Sub-columnas de coalición que se descartan (son parciales, no totales)
COLS_DESCARTAR = {
    "pvem, pt y morena", "pvem y pt", "pvem y morena", "pt y morena",
    "pan y pri",
    "pvem y morena",       # sub-columna en gubernatura (distinto de TOTAL PVEM Y MORENA)
}

# Columnas del esquema final (en orden)
COLS_FINALES = [
    "seccion", "casilla", "tipo_eleccion", "distrito",
    "PAN", "PRI", "PRD", "PVEM", "PT", "MC", "MORENA", "CAND_IND",
    "total_pvem_pt_morena", "total_pvem_morena", "total_pan_pri",
    "votos_validos", "cand_no_reg", "votos_nulos", "total_votos",
    "lista_nominal", "participacion",
    "anulada", "modificada",
]

In [ ]:
def normalizar_archivo(ruta, tipo_eleccion, distrito=None):
    """
    Lee un .xlsx, normaliza columnas y filtra filas de totales/resumen.

    Parámetros
    ----------
    ruta          : Path al archivo .xlsx
    tipo_eleccion : 'gubernatura' | 'diputaciones' | 'ayuntamiento'
    distrito      : str o None — e.g. 'DT-01', 'DTO-02'

    Retorna
    -------
    DataFrame normalizado con las columnas de COLS_FINALES que existan.
    """
    df = pd.read_excel(ruta)

    # 1. Eliminar columna índice artificial (#, Unnamed)
    df = df.drop(columns=[c for c in df.columns
                           if str(c).strip() in ("#", "Unnamed: 0")], errors="ignore")

    # 2. Renombrar usando el mapa (ignorando mayúsculas/minúsculas y espacios extra)
    rename = {
        col: RENAME_MAP[col.strip().lower()]
        for col in df.columns
        if col.strip().lower() in RENAME_MAP
    }
    df = df.rename(columns=rename)

    # 3. Descartar sub-columnas de coalición que no son totales
    df = df.drop(
        columns=[c for c in df.columns if c.strip().lower() in COLS_DESCARTAR],
        errors="ignore"
    )

    # 4. Filtrar filas de totales/resumen
    #    - Se conservan: filas con casilla definida (incluye VOTO ANTICIPADO)
    #    - Se eliminan: filas donde casilla es NaN (totales al pie de la tabla)
    df = df[df["casilla"].notna()].copy()

    # 5. Normalizar sección: rellenar NaN de VOTO ANTICIPADO con "0000"
    #    y asegurar formato de 4 dígitos con cero a la izquierda
    df["seccion"] = (
        df["seccion"]
        .fillna(0)
        .astype(float)
        .astype(int)
        .astype(str)
        .str.zfill(4)
    )

    # 6. Agregar columnas de metadatos
    df["tipo_eleccion"] = tipo_eleccion
    df["distrito"]      = distrito

    # 7. Reindexar al esquema final (columnas ausentes quedan como NaN)
    df = df.reindex(columns=COLS_FINALES)

    return df.reset_index(drop=True)

## 4. Carga y normalización de cada archivo

In [ ]:
FUENTES = [
    # (archivo,                                        tipo_eleccion,    distrito )
    ("cardenas_desglose_dt01_gubernatura.xlsx",        "gubernatura",    "DT-01"  ),
    ("cardenas_desglose_dt02_gubernatura.xlsx",        "gubernatura",    "DT-02"  ),
    ("cardenas_desglose_dt03_gubernatura.xlsx",        "gubernatura",    "DT-03"  ),
    ("cardenas_desglose_dto01_diputaciones.xlsx",      "diputaciones",   "DTO-01" ),
    ("cardenas_desglose_dto02_diputaciones.xlsx",      "diputaciones",   "DTO-02" ),
    ("cardenas_desglose_dto03_diputaciones.xlsx",      "diputaciones",   "DTO-03" ),
    ("cardenas_desglose_ayuntamientos_PELO2023-2024.xlsx", "ayuntamiento", None   ),
]

partes = []
for nombre, tipo, distrito in FUENTES:
    df_parte = normalizar_archivo(TABLAS / nombre, tipo, distrito)
    print(f"{nombre[:50]:<52}  →  {len(df_parte):>3} filas")
    partes.append(df_parte)

resultados = pd.concat(partes, ignore_index=True)
print(f"\nTotal consolidado: {len(resultados)} filas × {len(resultados.columns)} columnas")

## 5. Verificación de calidad

In [ ]:
resultados.head(3)

In [ ]:
# Distribución por tipo de elección y distrito
resultados.groupby(["tipo_eleccion", "distrito"]).size().rename("casillas")

In [ ]:
# Nulos por columna (solo columnas con al menos un nulo)
nulos = resultados.isna().sum()
nulos[nulos > 0]

In [ ]:
# Secciones únicas por tipo de elección
resultados.groupby("tipo_eleccion")["seccion"].nunique().rename("secciones_unicas")

In [ ]:
# Verificar que total_votos == suma de partidos (donde aplica)
COLS_PARTIDOS = ["PAN", "PRI", "PRD", "PVEM", "PT", "MC", "MORENA", "CAND_IND"]

# Para gubernatura, los votos de PVEM/PT/MORENA individuales + coalición pueden solaparse.
# Esta verificación es orientativa para diputaciones y ayuntamiento.
mask = resultados["tipo_eleccion"].isin(["diputaciones", "ayuntamiento"])
df_check = resultados[mask].copy()

df_check["suma_partidos"] = df_check[COLS_PARTIDOS].sum(axis=1, min_count=1)
df_check["diff"] = df_check["votos_validos"] - df_check["suma_partidos"]

print("Diferencia votos_validos − suma_partidos:")
print(df_check["diff"].describe().round(2))
print(f"\nFilas con diferencia > 1: {(df_check['diff'].abs() > 1).sum()}")

## 6. Exportar

In [ ]:
salida = PROCESADOS / "resultados_casilla.csv"
resultados.to_csv(salida, index=False, encoding="utf-8-sig")
print(f"Guardado: {salida}")
print(f"Shape   : {resultados.shape}")
resultados.dtypes